In [1]:
import pandas as pd
import numpy as np
import glob
import os

Uploading the upward generated timeseries:

In [2]:
upward_df = pd.read_csv("Data/upward.csv")

I look at the initial price at time step 0 and the final price of the stock to calculate the optimal portfolio value at final timestep and the optimal allocation for each timestep:

In [3]:
upward_df.loc[upward_df['tic'] == 'STOCK', ['date', 'tic', 'close']]

,date,tic,close
991,2023-01-10,STOCK,109
992,2023-01-11,STOCK,110
993,2023-01-12,STOCK,111
994,2023-01-13,STOCK,112
995,2023-01-14,STOCK,113
...,...,...,...
1977,2025-09-22,STOCK,1095
1978,2025-09-23,STOCK,1096
1979,2025-09-24,STOCK,1097
1980,2025-09-25,STOCK,1098


The stock starts from 110. I exclude the first price of 109 because the first initialization is fixed to 0.5, 0.5 that in the case of this experiment becomes 1,0, i.e. just cash. So then the porftolio value of the first timestep is fixed to 1000, then we can look at the stock price from the second timestep to see what could be the maximum achievable portfolio value. So given that the stock price starts at 110 and ends up at 1099, the optimal portfolio value would be:

In [4]:
C = 1000
weight = 1
R = 1099/110 

v = C * (weight * R + (1-weight)* 1)
v

9990.90909090909

I check that this final optimal portfolio value is the same for a run that I know it was the optimal allocation sequence and that the allocation weights are always 0,1 except for the first timestep, so that I can use this to as the optimal baseline to calculate the metrics later:

In [5]:
upward_run_work = pd.read_csv("Results/250619_hc_binary_udp/ddpg/upward/training_logs/01.csv")

In [6]:
optimal_upward_df_binary = upward_run_work.loc[(upward_run_work['episode'] == 50), ['day', 'allocation_weights', 'new_portfolio_value']]

In [7]:
assert (optimal_upward_df_binary['allocation_weights'].iloc[1:] == "0.00, 1.00").all()

In [8]:
optimal_upward_df_binary['new_portfolio_value'].iloc[-1] # final portfolio value as above

'9,990.91'

For the downward trend, it is trivial that the optimal portfolio value stays 1000:

In [9]:
C = 1000
weight = 0
R = 1/991

v = C * (weight * R + (1-weight)* 1)
v

1000.0

I construct the optimal downward df baseline as well (to be used in the metrics calculation):

In [10]:
optimal_downward_df_binary = upward_run_work.loc[(upward_run_work['episode'] == 50), ['day', 'allocation_weights', 'new_portfolio_value']]
optimal_downward_df_binary['new_portfolio_value'] = 1000
optimal_downward_df_binary["allocation_weights"] = [[1, 0]] * len(optimal_downward_df_binary)
optimal_downward_df_binary

,day,allocation_weights,new_portfolio_value
48510,1,"[1, 0]",1000
48511,2,"[1, 0]",1000
48512,3,"[1, 0]",1000
48513,4,"[1, 0]",1000
48514,5,"[1, 0]",1000
...,...,...,...
49495,986,"[1, 0]",1000
49496,987,"[1, 0]",1000
49497,988,"[1, 0]",1000
49498,989,"[1, 0]",1000


In [11]:
periodic_df = pd.read_csv("Data/periodic.csv")

In [12]:
periodic_df.loc[periodic_df['tic'] =='STOCK'].iloc[:5] 

,date,tic,close,return_t-1,return_t-2,return_t-3,ma_3,ma_5,ma_10,log_return,volatility_5,momentum_5
991,2023-01-10,STOCK,546.601954,0.005946,0.013791,0.023442,543.046487,538.290705,523.544972,0.005928,0.002549,24.853678
992,2023-01-11,STOCK,548.786168,0.003996,0.009966,0.017842,546.253094,542.401513,528.423588,0.003988,0.002785,20.554044
993,2023-01-12,STOCK,549.874749,0.001984,0.005988,0.011969,548.420957,545.560076,533.411063,0.001982,0.002969,15.792811
994,2023-01-13,STOCK,549.843251,-0.000057,0.001926,0.005930,549.501390,547.695457,537.648198,-0.000057,0.003099,10.676906
995,2023-01-14,STOCK,548.692382,-0.002093,-0.002150,-0.000171,549.470127,548.759701,541.039835,-0.002095,0.003177,5.321220


Here I mimic the logic of what would be written in the training_logs csv files if the optimal allocation would be chosen, to obtain the optimal baseline df for the periodic trend generator (look at timesteps_debug_periodic_alloc.txt to understand):


In [13]:
class PeriodicTrendPriceGenerator:
    def __init__(self, start=500, amplitude=50, frequency=0.15):
        self.start = start
        self.amplitude = amplitude
        self.frequency = frequency
        self.t = 0

    def generate_price(self, last_price=None):
        value = self.amplitude * np.sin(self.frequency * self.t) + self.start
        self.t += 1
        return value

# Generate 991 days of stock prices
days = 999
generator = PeriodicTrendPriceGenerator(start= 500, amplitude=50, frequency=0.15)
stock_prices = [generator.generate_price() for _ in range(days)]

stock_prices = np.array(stock_prices)[8:]
allocation = [1,0]
portfolio_value = 1000
days_df = []
optimal_allocations = []
optimal_portfolio_values = []

for day, stock_price in enumerate(stock_prices):
    print("day: ", day)
    if day == 0:
        print("stock price today", stock_prices[day])
        print("stock price yesterday", stock_prices[day-1])
        pass
    else:
        if day == 1:
           
            assert round(stock_prices[day]) == 549
        old_allocation = allocation
        
        print("stock price today", stock_prices[day])
        print("stock price yesterday", stock_prices[day-1])
        portfolio_return = sum((np.array([500, stock_prices[day]]) / np.array([500, stock_prices[day-1]])-1)*np.array(old_allocation))
        new_portfolio_value = portfolio_value*(1+portfolio_return)
        portfolio_value = new_portfolio_value
        print("portfolio_value", portfolio_value)
        if (stock_prices[day] > stock_prices[day-1]):
            allocation = [0, 1]
        else:
            allocation = [1, 0]
        print("old_allocation", old_allocation)
        print("allocation:", allocation)
        days_df.append(day)
        optimal_allocations.append(old_allocation)
        optimal_portfolio_values.append(portfolio_value)

optimal_periodic_df_binary = pd.DataFrame({"day": days_df,
                                            "allocation_weights":optimal_allocations,
                                            "new_portfolio_value": optimal_portfolio_values
                                           })

day:  0
stock price today 546.6019542983613
stock price yesterday 455.52048593604115
day:  1
stock price today 548.786167891333
stock price yesterday 546.6019542983613
portfolio_value 1000.0
old_allocation [1, 0]
allocation: [0, 1]
day:  2
stock price today 549.8747493302027
stock price yesterday 548.786167891333
portfolio_value 1001.9836167574204
old_allocation [0, 1]
allocation: [0, 1]
day:  3
stock price today 549.843251422696
stock price yesterday 549.8747493302027
portfolio_value 1001.9262211644743
old_allocation [0, 1]
allocation: [1, 0]
day:  4
stock price today 548.6923815439097
stock price yesterday 549.843251422696
portfolio_value 1001.9262211644743
old_allocation [1, 0]
allocation: [1, 0]
day:  5
stock price today 546.4479857501934
stock price yesterday 548.6923815439097
portfolio_value 1001.9262211644743
old_allocation [1, 0]
allocation: [1, 0]
day:  6
stock price today 543.1604683324437
stock price yesterday 546.4479857501934
portfolio_value 1001.9262211644743
old_allocati

In [14]:
optimal_periodic_df_binary 

,day,allocation_weights,new_portfolio_value
0,1,"[1, 0]",1000.000000
1,2,"[0, 1]",1001.983617
2,3,"[0, 1]",1001.926221
3,4,"[1, 0]",1001.926221
4,5,"[1, 0]",1001.926221
...,...,...,...
985,986,"[1, 0]",95929.842356
986,987,"[1, 0]",95929.842356
987,988,"[1, 0]",95929.842356
988,989,"[0, 1]",96323.358112


I clean the allocation weights column so that is always a list of two numbers, and the portfolio is always a float with two decimal digits:

In [15]:
def clean_allocation_df(df):
    allocation_column = df.filter(regex = 'allocation').columns[0]
    portfolio_column = df.filter(regex = 'portfolio').columns[0]
    # Convert allocation_weights to [1, 0] or [0, 1] format
    if df[allocation_column].dtype == object:
        df[allocation_column] = df[allocation_column].apply(
        lambda x: [int(i) for i in x] if isinstance(x, (list, tuple)) else [int(float(i.strip())) for i in str(x).split(",")]
    )

    # Clean and convert new_portfolio_value to float with 2 decimals
    df[portfolio_column] = (
        df[portfolio_column]
        .astype(str)                      # ensure string for cleanup
        .str.replace(",", "", regex=False)  # remove thousand separators
        .astype(float)
        .round(2)
    )

    return df

optimal_upward_df_binary = clean_allocation_df(optimal_upward_df_binary)
optimal_periodic_df_binary = clean_allocation_df(optimal_periodic_df_binary)
optimal_downward_df_binary = clean_allocation_df(optimal_downward_df_binary)

In [16]:
optimal_upward_df_binary.head()

,day,allocation_weights,new_portfolio_value
48510,1,"[1, 0]",1000.00
48511,2,"[0, 1]",1009.09
48512,3,"[0, 1]",1018.18
48513,4,"[0, 1]",1027.27
48514,5,"[0, 1]",1036.36


Now I can compute the Mean allocation error and the difference of the final portfolio value between each single run and the optimal baseline (according to the right generator associated to the run):

In [17]:
base_path = "Results/250619_hc_binary_udp"
base_path_noisy = "Results/250619_hc_binary_udp_noise"
agents = ["a2c", "ddpg", "ppo"]
regimes = ["upward", "downward", "periodic", "upward_noise", "downward_noise", "periodic_noise"]

# Result containers
result_dfs = {}  # structure: result_dfs[agent][regime]
all_data = {}

# Utility function to clean 'allocation_weights'
def clean_allocation_df(df):
    df = df.copy()
    df["allocation_weights"] = df["allocation_weights"].apply(
        lambda x: [float(i) for i in str(x).strip("[]").split(",")]
    )
    return df

# Function to compute mean L2 error
def compute_mean_l2_error(agent_allocs, opt_allocs):
    errors = np.linalg.norm(agent_allocs - opt_allocs, axis=1)
    return np.mean(errors)

# You must define or load the optimal allocation DataFrames for each regime
optimal_data = {
    "upward": clean_allocation_df(optimal_upward_df_binary),
    "downward": clean_allocation_df(optimal_downward_df_binary),
    "periodic": clean_allocation_df(optimal_periodic_df_binary),
     "upward_noise": clean_allocation_df(optimal_upward_df_binary),
    "downward_noise": clean_allocation_df(optimal_downward_df_binary),
    "periodic_noise": clean_allocation_df(optimal_periodic_df_binary),
}

for agent in agents:
    result_dfs[agent] = {}
    all_data[agent] = {}
    
    for regime in regimes:
        # Prepare optimal allocations
        opt_allocs_array = np.array(optimal_data[regime]["allocation_weights"].tolist())

        # Gather CSVs
        # Select correct base path based on regime type
        current_base = base_path_noisy if "noise" in regime else base_path
        csv_pattern = os.path.join(current_base, agent, regime, "training_logs", "*.csv")

        csv_files = sorted(glob.glob(csv_pattern))
        print(csv_files)
        mean_errors = []
        dfs = []
        delta_vs = []

        for fpath in csv_files:
            df = pd.read_csv(fpath)
            # taking episode 50, that is the last complete episode:
            df = df.loc[df['episode'] == 50, ['day', 'allocation_weights', 'new_portfolio_value']]
            df = clean_allocation_df(df)

            agent_allocs_array = np.array(df["allocation_weights"].tolist())
            min_len = min(len(agent_allocs_array), len(opt_allocs_array))
            opt_df = optimal_data[regime]

            error = compute_mean_l2_error(agent_allocs_array[:min_len], opt_allocs_array[:min_len])
            mean_errors.append(error)
            dfs.append(df)
              # Compute ∆V
            try:
                V_agent_T = float(str(df["new_portfolio_value"].iloc[-1]).replace(",", ""))
                V_opt_T = float(str(opt_df["new_portfolio_value"].iloc[min_len - 1]).replace(",", ""))
                delta_v = V_agent_T - V_opt_T
            except Exception as e:
                delta_v = np.nan  # fallback if any issue
            delta_vs.append(delta_v)

        result_dfs[agent][regime] = pd.DataFrame({
            "run": list(range(1, len(mean_errors) + 1)),
            "mean_allocation_error": mean_errors,
            "delta_V": delta_vs
        })
        all_data[agent][regime] = pd.concat(dfs, ignore_index=True)

# Example: access PPO's results on periodic regime
print(result_dfs["ppo"]["periodic"])


['Results/250619_hc_binary_udp/a2c/upward/training_logs/01.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/02.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/03.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/04.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/05.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/06.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/07.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/08.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/09.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/10.csv']
['Results/250619_hc_binary_udp/a2c/downward/training_logs/01.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/02.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/03.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/04.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/05.csv', 'Results/250619_hc_binary_ud

This is to access the mean allocation error and the delta V for each run after choosing an agent and a generator:

In [18]:
result_dfs['ppo']['downward_noise'] 

,run,mean_allocation_error,delta_V
0,1,0.451406,-408.83
1,2,0.564257,3.64
2,3,0.631396,128.05
3,4,0.282843,310.00
4,5,0.368553,-146.14
5,6,0.435692,42.19
6,7,0.269986,107.98
7,8,0.324269,34.04
8,9,0.511402,-105.38
9,10,0.402837,193.06


Generate the LATEX table:

In [24]:
for agent in ["a2c", "ddpg", "ppo"]:
    print("\\begin{table}[ht]")
    print("  \\centering")
    print(f"  \\caption{{Expt. 3 - {agent.upper()} Results Averaged Across 10 Runs with Different Generators}}")
    print(f"  \\label{{tab:{agent}_results}}")

    # --- Clean Regimes Subtable ---
    print("  \\begin{subtable}[t]{0.48\\textwidth}")
    print("  \\centering")
    print("  \\caption{Generators with no Noise}")
    print("  \\begin{tabular}{ccc}")
    print("  \\toprule")
    print("  Generator & Mean Allocation Error & $\\Delta V$ \\\\")
    print("  \\midrule")
    for regime in ["upward", "downward", "periodic"]:
        df = result_dfs[agent].get(regime)
        if df is not None and not df.empty:
            mean_error = df["mean_allocation_error"].mean() # to calculate the average Mean Allocation Error across runs
            delta_v = df["delta_V"].mean() # to calculate the average difference in final portfolio value across runs 
            print(f"  {regime.capitalize()} & {mean_error:.2f} & {delta_v:.2f} \\\\")
    print("  \\bottomrule")
    print("  \\end{tabular}")
    print("  \\end{subtable}")

    print("  \\hfill")

    # --- Noisy Regimes Subtable ---
    print("  \\begin{subtable}[t]{0.48\\textwidth}")
    print("  \\centering")
    print("  \\caption{Generators with Noise}")
    print("  \\begin{tabular}{ccc}")
    print("  \\toprule")
    print("  Generator & Mean Allocation Error & $\\Delta V$ \\\\")
    print("  \\midrule")
    for regime in ["upward_noise", "downward_noise", "periodic_noise"]:
        df = result_dfs[agent].get(regime)
        if df is not None and not df.empty:
            mean_error = df["mean_allocation_error"].mean()
            delta_v = df["delta_V"].mean() #np.mean(np.abs(df["delta_V"]))
            print(f"  {regime.replace('_noise','').capitalize()} (Noise) & {mean_error:.2f} & {delta_v:.2f} \\\\")
    print("  \\bottomrule")
    print("  \\end{tabular}")
    print("  \\end{subtable}")

    print("\\end{table}")
    print("\n\n")

\begin{table}[ht]
  \centering
  \caption{Expt. 3 - A2C Results Averaged Across 10 Runs with Different Generators}
  \label{tab:a2c_results}
  \begin{subtable}[t]{0.48\textwidth}
  \centering
  \caption{Generators with no Noise}
  \begin{tabular}{ccc}
  \toprule
  Generator & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward & 0.06 & -924.72 \\
  Downward & 0.01 & -70.81 \\
  Periodic & 0.47 & -75042.38 \\
  \bottomrule
  \end{tabular}
  \end{subtable}
  \hfill
  \begin{subtable}[t]{0.48\textwidth}
  \centering
  \caption{Generators with Noise}
  \begin{tabular}{ccc}
  \toprule
  Generator & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward (Noise) & 0.09 & -6667.27 \\
  Downward (Noise) & 0.20 & -92.60 \\
  Periodic (Noise) & 0.72 & -95129.39 \\
  \bottomrule
  \end{tabular}
  \end{subtable}
\end{table}



\begin{table}[ht]
  \centering
  \caption{Expt. 3 - DDPG Results Averaged Across 10 Runs with Different Generators}
  \label{tab:ddpg_results}
  \begin{subtable}[t]{

Creating a table with only the best agents for each generator case:

In [22]:
# Define which agent is best for each regime
best_agent_per_regime = {
    "upward": "ppo",
    "downward": "a2c",
    "periodic": "a2c",
    "upward_noise": "a2c",
    "downward_noise": "ppo",
    "periodic_noise": "ppo"
}

# Define the order and labels for display
regimes_order = [
    ("upward", "Upward"),
    ("downward", "Downward"),
    ("periodic", "Periodic"),
    ("upward_noise", "Upward (Noise)"),
    ("downward_noise", "Downward (Noise)"),
    ("periodic_noise", "Periodic (Noise)")
]

# Start LaTeX table
print("\\begin{table}[ht]")
print("  \\centering")
print("  \\caption{Expt. 3 -- Best Agent per Generator}")
print("  \\label{tab:best_agent_results}")
print("  \\begin{tabular}{lccc}")
print("  \\toprule")
print("  Generator & Best Agent & Mean Allocation Error & $\\Delta V$ \\\\")
print("  \\midrule")

# Fill in table rows
for regime_key, label in regimes_order:
    best_agent = best_agent_per_regime[regime_key]
    df = result_dfs[best_agent].get(regime_key)
    if df is not None and not df.empty:
        mean_error = df["mean_allocation_error"].mean()
        delta_v = df["delta_V"].mean()
        print(f"  {label} & {best_agent.upper()} & {mean_error:.2f} & {delta_v:.2f} \\\\")

print("  \\bottomrule")
print("  \\end{tabular}")
print("\\end{table}")


\begin{table}[ht]
  \centering
  \caption{Expt. 3 -- Best Agent per Generator}
  \label{tab:best_agent_results}
  \begin{tabular}{lccc}
  \toprule
  Generator & Best Agent & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward & PPO & 0.04 & -572.12 \\
  Downward & A2C & 0.01 & -70.81 \\
  Periodic & A2C & 0.47 & -75042.38 \\
  Upward (Noise) & A2C & 0.09 & -6667.27 \\
  Downward (Noise) & PPO & 0.42 & 15.86 \\
  Periodic (Noise) & PPO & 0.65 & -92974.54 \\
  \bottomrule
  \end{tabular}
\end{table}
